# model1
## score: .84


In [ ]:
import json
import re
import shutil
from pathlib import Path

import torch
from safetensors import safe_open
from safetensors.torch import save_file

BASE_MODEL_NAME = "nvidia/NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"

SOURCE_ADAPTER_FALLBACK = Path(
    "/kaggle/input/models/huikang/nemotron-adapter/transformers/default/20"
)
WORK_DIR = Path("/kaggle/working")
OUTPUT_ROOT = WORK_DIR / "adapter_candidates"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

ENABLE_SHORT_SFT = False
SFT_MAX_STEPS = 800
SFT_SUBSAMPLE = 2400
SFT_MAX_SEQ_LEN = 1024
SFT_PER_DEVICE_BS = 1
SFT_GRAD_ACCUM = 8
SFT_LR = 8e-5
SFT_WARMUP_RATIO = 0.05
SFT_SEED = 42



ENABLE_DPO = False
DPO_INCLUDE_GREEDY = True
DPO_ROLLOUT_PROMPTS = 220
DPO_SAMPLES_PER_PROMPT = 4
DPO_MAX_NEW_TOKENS = 448
DPO_GEN_TEMP = 0.78
DPO_MAX_STEPS = 360
DPO_GRAD_ACCUM = 8
DPO_LR = 1e-6
DPO_BETA = 0.05
DPO_WARMUP_RATIO = 0.05
DPO_SEED = 42


def resolve_source_adapter_path() -> Path:
    root = Path(
        "/kaggle/input/models/huikang/nemotron-adapter/transformers/default"
    )
    best_path = None
    best_ver = -1
    if root.is_dir():
        for child in root.iterdir():
            if not child.is_dir() or not child.name.isdigit():
                continue
            if (child / "adapter_model.safetensors").is_file() and (
                child / "adapter_config.json"
            ).is_file():
                ver = int(child.name)
                if ver > best_ver:
                    best_ver = ver
                    best_path = child
    if best_path is not None:
        return best_path
    if SOURCE_ADAPTER_FALLBACK.is_dir():
        return SOURCE_ADAPTER_FALLBACK
    return SOURCE_ADAPTER_FALLBACK


def _resolve_train_csv() -> Path:
    for p in (
        Path("/kaggle/input/nvidia-nemotron-3-reasoning-challenge/train.csv"),
        Path("/kaggle/input/competitions/nvidia-nemotron-model-reasoning-challenge/train.csv"),
    ):
        if p.is_file():
            return p
    raise FileNotFoundError("train.csv not found")


def _resolve_base_model_dir() -> Path:
    root = Path("/kaggle/input/models/metric/nemotron-3-nano-30b-a3b-bf16/transformers/default/1")
    nested = root / "NVIDIA-Nemotron-3-Nano-30B-A3B-BF16"
    if nested.is_dir() and (nested / "config.json").is_file():
        return nested
    if root.is_dir() and (root / "config.json").is_file():
        return root
    raise FileNotFoundError("Nemotron base not found")


def _clear_merge_weights() -> None:
    import gc

    for name in (
        "source_adapter_tensors",
        "source_base_names",
        "mamba_merge_layers",
        "mamba_merge_bases",
    ):
        g = globals()
        if name in g:
            del g[name]
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def run_short_sft(selected_manifest: dict, submission_zip: Path) -> None:
    import gc
    import os
    import random
    import sys

    import pandas as pd
    from datasets import Dataset
    from peft import PeftModel
    from transformers import (
        AutoModelForCausalLM,
        AutoTokenizer,
        Trainer,
        TrainingArguments,
    )

    if not torch.cuda.is_available():
        raise RuntimeError("ENABLE_SHORT_SFT needs a GPU kernel")

    _clear_merge_weights()

    train_path = _resolve_train_csv()
    base_dir = _resolve_base_model_dir()
    merged_dir = Path(selected_manifest["output_dir"])
    if not (merged_dir / "adapter_config.json").is_file():
        raise FileNotFoundError(merged_dir)

    random.seed(SFT_SEED)
    df = pd.read_csv(train_path)
    df = df.dropna(subset=["prompt", "answer"])
    df["prompt"] = df["prompt"].astype(str).str.strip()
    df["answer"] = df["answer"].astype(str).str.strip()
    df = df[(df["prompt"].str.len() > 0) & (df["answer"].str.len() > 0)]
    n = min(SFT_SUBSAMPLE, len(df))
    if n < len(df):
        df = df.sample(n=n, random_state=SFT_SEED)

    rows = []
    for _, r in df.iterrows():
        p, a = r["prompt"], r["answer"]
        user_msg = (
            f"{p}\n\n"
            "Solve this step by step, then put your final answer in \\boxed{}."
        )
        assistant_msg = f"The answer is {a}.\n\\boxed{{{a}}}"
        rows.append({"user": user_msg, "assistant": assistant_msg})

    raw_ds = Dataset.from_list(rows)

    tokenizer = AutoTokenizer.from_pretrained(
        str(base_dir),
        trust_remote_code=True,
        local_files_only=True,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    def tokenize_batch(examples):
        out_ids, out_mask, out_labels = [], [], []
        for ut, at in zip(examples["user"], examples["assistant"]):
            messages = [
                {"role": "user", "content": ut},
                {"role": "assistant", "content": at},
            ]
            try:
                full_text = tokenizer.apply_chat_template(
                    messages,
                    tokenize=False,
                    add_generation_prompt=False,
                    chat_template_kwargs={"enable_thinking": False},
                )
                prefix_text = tokenizer.apply_chat_template(
                    [{"role": "user", "content": ut}],
                    tokenize=False,
                    add_generation_prompt=True,
                    chat_template_kwargs={"enable_thinking": False},
                )
            except TypeError:
                full_text = tokenizer.apply_chat_template(
                    messages,
                    tokenize=False,
                    add_generation_prompt=False,
                )
                prefix_text = tokenizer.apply_chat_template(
                    [{"role": "user", "content": ut}],
                    tokenize=False,
                    add_generation_prompt=True,
                )

            full_enc = tokenizer(
                full_text,
                truncation=True,
                max_length=SFT_MAX_SEQ_LEN,
                padding="max_length",
            )
            pref_enc = tokenizer(
                prefix_text,
                truncation=True,
                max_length=SFT_MAX_SEQ_LEN,
                padding=False,
            )
            lab = list(full_enc["input_ids"])
            pref_len = min(len(pref_enc["input_ids"]), len(lab))
            for i in range(pref_len):
                lab[i] = -100
            for i, m in enumerate(full_enc["attention_mask"]):
                if m == 0:
                    lab[i] = -100
            out_ids.append(full_enc["input_ids"])
            out_mask.append(full_enc["attention_mask"])
            out_labels.append(lab)

        return {
            "input_ids": out_ids,
            "attention_mask": out_mask,
            "labels": out_labels,
        }

    tok_ds = raw_ds.map(tokenize_batch, batched=True, remove_columns=raw_ds.column_names)

    for name, mod in sys.modules.items():
        if "modeling_nemotron_h" in name:
            mod.is_fast_path_available = False

    model = AutoModelForCausalLM.from_pretrained(
        str(base_dir),
        device_map={"": 0},
        trust_remote_code=True,
        dtype=torch.bfloat16,
        local_files_only=True,
    )
    model.gradient_checkpointing_enable()
    model.enable_input_require_grads()
    model = PeftModel.from_pretrained(model, str(merged_dir), is_trainable=True)

    try:
        import triton.backends.nvidia.compiler as nv_compiler

        os.makedirs("/tmp/ptxas-blackwell", exist_ok=True)
        os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = "/tmp/ptxas-blackwell"
        nv_compiler.get_ptxas_version = lambda arch: "12.0"
    except Exception:
        pass

    use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    sft_out = WORK_DIR / "sft_adapter"
    if sft_out.is_dir():
        shutil.rmtree(sft_out)
    sft_out.mkdir(parents=True, exist_ok=True)

    train_args = TrainingArguments(
        output_dir=str(WORK_DIR / "sft_trainer_state"),
        max_steps=SFT_MAX_STEPS,
        per_device_train_batch_size=SFT_PER_DEVICE_BS,
        gradient_accumulation_steps=SFT_GRAD_ACCUM,
        learning_rate=SFT_LR,
        warmup_ratio=SFT_WARMUP_RATIO,
        lr_scheduler_type="cosine",
        weight_decay=0.01,
        bf16=use_bf16,
        fp16=not use_bf16,
        logging_steps=10,
        save_strategy="no",
        report_to="none",
        max_grad_norm=1.0,
        dataloader_pin_memory=False,
    )

    def collate(features):
        return {
            "input_ids": torch.tensor([f["input_ids"] for f in features], dtype=torch.long),
            "attention_mask": torch.tensor([f["attention_mask"] for f in features], dtype=torch.long),
            "labels": torch.tensor([f["labels"] for f in features], dtype=torch.long),
        }

    trainer = Trainer(
        model=model,
        args=train_args,
        train_dataset=tok_ds,
        data_collator=collate,
    )
    trainer.train()
    model.save_pretrained(str(sft_out))

    zip_base = WORK_DIR / "submission_sft"
    zip_path = Path(shutil.make_archive(str(zip_base), "zip", sft_out))
    shutil.copy2(zip_path, submission_zip)

    meta = {
        "merged_adapter_dir": str(merged_dir),
        "sft_adapter_dir": str(sft_out),
        "max_steps": SFT_MAX_STEPS,
        "subsample_rows": int(n),
    }
    with open(WORK_DIR / "sft_meta.json", "w") as f:
        json.dump(meta, f, indent=2)
        f.write("\n")

    del trainer, model, tok_ds, raw_ds, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()




def run_dpo_from_rollouts(selected_manifest: dict, submission_zip: Path) -> None:
    import gc
    import importlib.util
    import os
    import random
    import sys

    import pandas as pd
    import torch
    from datasets import Dataset
    from peft import PeftModel
    from transformers import AutoModelForCausalLM, AutoTokenizer

    if importlib.util.find_spec("trl") is None:
        raise RuntimeError("Install trl for DPO: pip install trl")

    from trl import DPOConfig, DPOTrainer

    if not torch.cuda.is_available():
        raise RuntimeError("ENABLE_DPO needs a GPU kernel")

    _clear_merge_weights()

    train_path = _resolve_train_csv()
    base_dir = _resolve_base_model_dir()
    merged_dir = Path(selected_manifest["output_dir"])
    if not (merged_dir / "adapter_config.json").is_file():
        raise FileNotFoundError(merged_dir)

    random.seed(DPO_SEED)
    torch.manual_seed(DPO_SEED)

    df = pd.read_csv(train_path)
    df = df.dropna(subset=["prompt", "answer"])
    df["prompt"] = df["prompt"].astype(str).str.strip()
    df["answer"] = df["answer"].astype(str).str.strip()
    df = df[(df["prompt"].str.len() > 0) & (df["answer"].str.len() > 0)]
    n_roll = min(DPO_ROLLOUT_PROMPTS, len(df))
    if n_roll < len(df):
        df = df.sample(n=n_roll, random_state=DPO_SEED)

    boxed_re = re.compile(r"\\boxed\{([^}]*)\}")

    def norm_ans(s: str) -> str:
        t = str(s).strip().replace("$", "").replace(" ", "")
        return t

    def extract_boxed(text: str) -> str | None:
        if not text:
            return None
        ms = boxed_re.findall(text)
        if not ms:
            return None
        return ms[-1].strip()

    def score_completion(text: str, gold: str) -> int:
        ext = extract_boxed(text)
        if ext is None:
            return 0
        if norm_ans(ext) == norm_ans(gold):
            return 2
        try:
            g = float(str(gold).replace(" ", ""))
            if abs(float(ext) - g) < 1e-6 * max(1.0, abs(g)):
                return 1
        except ValueError:
            pass
        return 0

    tokenizer = AutoTokenizer.from_pretrained(
        str(base_dir),
        trust_remote_code=True,
        local_files_only=True,
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    tokenizer.padding_side = "left"

    for name, mod in sys.modules.items():
        if "modeling_nemotron_h" in name:
            mod.is_fast_path_available = False

    model = AutoModelForCausalLM.from_pretrained(
        str(base_dir),
        device_map={"": 0},
        trust_remote_code=True,
        dtype=torch.bfloat16,
        local_files_only=True,
    )
    model.gradient_checkpointing_enable()
    model.enable_input_require_grads()
    model = PeftModel.from_pretrained(model, str(merged_dir), is_trainable=True)

    try:
        import triton.backends.nvidia.compiler as nv_compiler

        os.makedirs("/tmp/ptxas-blackwell", exist_ok=True)
        os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = "/tmp/ptxas-blackwell"
        nv_compiler.get_ptxas_version = lambda arch: "12.0"
    except Exception:
        pass

    device = next(model.parameters()).device

    def build_user_msg(p: str) -> str:
        return (
            f"{p}\n\n"
            "Solve this step by step, then put your final answer in \\boxed{}."
        )

    def encode_prompt(user_msg: str):
        try:
            enc = tokenizer.apply_chat_template(
                [{"role": "user", "content": user_msg}],
                tokenize=True,
                add_generation_prompt=True,
                return_dict=True,
                return_tensors="pt",
                chat_template_kwargs={"enable_thinking": False},
            )
        except TypeError:
            enc = tokenizer.apply_chat_template(
                [{"role": "user", "content": user_msg}],
                tokenize=True,
                add_generation_prompt=True,
                return_dict=True,
                return_tensors="pt",
            )
        return {k: v.to(device) for k, v in enc.items()}

    pairs: list[dict[str, str]] = []
    model.eval()
    for _, r in df.iterrows():
        p, gold = r["prompt"], r["answer"]
        user_msg = build_user_msg(p)
        enc = encode_prompt(user_msg)
        prompt_len = int(enc["input_ids"].shape[1])
        completions: list[tuple[str, int]] = []
        base_ids = enc["input_ids"].clone()
        base_mask = enc["attention_mask"].clone()
        with torch.inference_mode():
            if DPO_INCLUDE_GREEDY:
                out = model.generate(
                    input_ids=base_ids,
                    attention_mask=base_mask,
                    max_new_tokens=DPO_MAX_NEW_TOKENS,
                    do_sample=False,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                )
                gen_ids = out[0, prompt_len:]
                text = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
                completions.append((text, score_completion(text, gold)))
            for _ in range(DPO_SAMPLES_PER_PROMPT):
                out = model.generate(
                    input_ids=base_ids,
                    attention_mask=base_mask,
                    max_new_tokens=DPO_MAX_NEW_TOKENS,
                    do_sample=True,
                    temperature=DPO_GEN_TEMP,
                    top_p=0.9,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id,
                )
                gen_ids = out[0, prompt_len:]
                text = tokenizer.decode(gen_ids, skip_special_tokens=True).strip()
                completions.append((text, score_completion(text, gold)))

        uniq: dict[str, int] = {}
        for t, s in completions:
            if t not in uniq or s > uniq[t]:
                uniq[t] = s
        completions = [(t, uniq[t]) for t in uniq]

        if len(completions) < 2:
            continue
        best = max(completions, key=lambda x: (x[1], -len(x[0])))
        worst = min(completions, key=lambda x: (x[1], len(x[0])))
        if best[0] == worst[0]:
            continue
        chosen_text, rejected_text = best[0], worst[0]
        pairs.append(
            {"prompt": user_msg, "chosen": chosen_text, "rejected": rejected_text}
        )

    if len(pairs) < 16:
        raise RuntimeError(
            f"Too few DPO pairs ({len(pairs)}); increase prompts/samples or relax filters."
        )

    dpo_ds = Dataset.from_list(pairs)
    model.train()

    use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
    dpo_out = WORK_DIR / "dpo_adapter"
    if dpo_out.is_dir():
        shutil.rmtree(dpo_out)
    dpo_out.mkdir(parents=True, exist_ok=True)

    dpo_args = DPOConfig(
        output_dir=str(WORK_DIR / "dpo_trainer_state"),
        beta=DPO_BETA,
        learning_rate=DPO_LR,
        max_steps=DPO_MAX_STEPS,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=DPO_GRAD_ACCUM,
        warmup_ratio=DPO_WARMUP_RATIO,
        lr_scheduler_type="cosine",
        weight_decay=0.0,
        bf16=use_bf16,
        fp16=not use_bf16,
        logging_steps=5,
        save_strategy="no",
        report_to="none",
        max_grad_norm=1.0,
        max_prompt_length=min(1024, getattr(tokenizer, "model_max_length", 1024) or 1024),
        max_completion_length=DPO_MAX_NEW_TOKENS + 64,
    )

    kwargs = {"model": model, "ref_model": None, "args": dpo_args, "train_dataset": dpo_ds}
    try:
        trainer = DPOTrainer(**kwargs, processing_class=tokenizer)
    except TypeError:
        trainer = DPOTrainer(**kwargs, tokenizer=tokenizer)

    trainer.train()
    model.save_pretrained(str(dpo_out))

    zip_base = WORK_DIR / "submission_dpo"
    zip_path = Path(shutil.make_archive(str(zip_base), "zip", dpo_out))
    shutil.copy2(zip_path, submission_zip)

    meta = {
        "merged_adapter_dir": str(merged_dir),
        "dpo_adapter_dir": str(dpo_out),
        "dpo_pairs": len(pairs),
        "max_steps": DPO_MAX_STEPS,
    }
    with open(WORK_DIR / "dpo_meta.json", "w") as f:
        json.dump(meta, f, indent=2)
        f.write("\n")

    del trainer, model, dpo_ds, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()



FORCED_FUSED_RANK = 32
MERGE_SELECTION = "single"
MERGE_SHORTLIST = [
    "block_topk_floor4_model",
    "block_topk_floor4_x_bias_model",
    "block_topk_floor4_gate_bias_model",
    "block_topk_floor4_sym_bias_model",
    "block_topk_floor6_model",
]

CANDIDATE_SPECS = [
    {
        "name": "dense_svd_model",
        "strategy": "dense_svd",
        "min_rank_per_component": 0,
        "component_priority": {},
        "drop_lm_head": False,
        "namespace_mode": "model",
    },
    {
        "name": "block_topk_floor4_model",
        "strategy": "block_topk",
        "min_rank_per_component": 4,
        "component_priority": {},
        "drop_lm_head": False,
        "namespace_mode": "model",
    },
    {
        "name": "block_topk_floor4_x_bias_model",
        "strategy": "block_topk",
        "min_rank_per_component": 4,
        "component_priority": {"x_proj": 1.15},
        "drop_lm_head": False,
        "namespace_mode": "model",
    },
    {
        "name": "block_topk_floor4_gate_bias_model",
        "strategy": "block_topk",
        "min_rank_per_component": 4,
        "component_priority": {"gate_proj": 1.15},
        "drop_lm_head": False,
        "namespace_mode": "model",
    },
    {
        "name": "block_topk_floor4_sym_bias_model",
        "strategy": "block_topk",
        "min_rank_per_component": 4,
        "component_priority": {"gate_proj": 1.1, "x_proj": 1.1},
        "drop_lm_head": False,
        "namespace_mode": "model",
    },
    {
        "name": "block_topk_floor6_model",
        "strategy": "block_topk",
        "min_rank_per_component": 6,
        "component_priority": {},
        "drop_lm_head": False,
        "namespace_mode": "model",
    },
    {
        "name": "block_topk_floor4_model_no_lm_head",
        "strategy": "block_topk",
        "min_rank_per_component": 4,
        "component_priority": {},
        "drop_lm_head": True,
        "namespace_mode": "model",
    },
    {
        "name": "block_topk_floor4_backbone",
        "strategy": "block_topk",
        "min_rank_per_component": 4,
        "component_priority": {},
        "drop_lm_head": False,
        "namespace_mode": "backbone",
    },
]

ACTIVE_CANDIDATE_NAME = "block_topk_floor4_model"
CANDIDATE_BY_NAME = {spec["name"]: spec for spec in CANDIDATE_SPECS}
ACTIVE_CANDIDATE = CANDIDATE_BY_NAME[ACTIVE_CANDIDATE_NAME]

SOURCE_ADAPTER_PATH = resolve_source_adapter_path()

with open(SOURCE_ADAPTER_PATH / "adapter_config.json") as f:
    source_adapter_config = json.load(f)

source_adapter_tensors = {}
with safe_open(
    SOURCE_ADAPTER_PATH / "adapter_model.safetensors",
    framework="pt",
    device="cpu",
) as f:
    for key in f.keys():
        source_adapter_tensors[key] = f.get_tensor(key)

source_base_names = sorted(
    {
        re.sub(r"\.lora_[AB]\.weight$", "", key)
        for key in source_adapter_tensors
    }
)

mamba_merge_layers = {}
for base_name in source_base_names:
    for proj_name in ("gate_proj", "x_proj"):
        if f".{proj_name}" in base_name:
            layer_prefix = base_name.rsplit(f".{proj_name}", 1)[0]
            mamba_merge_layers.setdefault(layer_prefix, {})[proj_name] = base_name

mamba_merge_bases = {
    item
    for layer in mamba_merge_layers.values()
    for item in layer.values()
}



def rename_base_name(base_name: str, namespace_mode: str) -> str:
    if namespace_mode == "model":
        return base_name
    if namespace_mode == "backbone":
        return base_name.replace(
            "base_model.model.model",
            "base_model.model.backbone",
        )
    raise ValueError(f"Unknown namespace_mode: {namespace_mode}")


def rewrite_expert_base_name(base_name: str, expert_idx: int) -> str:
    if ".experts.w1" in base_name:
        return re.sub(
            r"\.experts\.w1",
            f".experts.{expert_idx}.up_proj",
            base_name,
        )
    if ".experts.w2" in base_name:
        return re.sub(
            r"\.experts\.w2",
            f".experts.{expert_idx}.down_proj",
            base_name,
        )
    raise ValueError(f"Unsupported expert base name: {base_name}")


def compute_target_modules(tensors: dict) -> list[str]:
    target_modules = sorted(
        {
            key.removesuffix(".lora_A.weight").removesuffix(".lora_B.weight").rsplit(".", 1)[-1]
            for key in tensors
        }
    )
    return target_modules


def build_adapter_config(
    output_tensors: dict,
    candidate_spec: dict,
) -> dict:
    config = dict(source_adapter_config)
    config["base_model_name_or_path"] = BASE_MODEL_NAME
    config["inference_mode"] = True
    config["target_modules"] = compute_target_modules(output_tensors)
    config["r"] = FORCED_FUSED_RANK
    config["lora_alpha"] = FORCED_FUSED_RANK
    config["rank_pattern"] = {}
    config["alpha_pattern"] = {}
    config["task_type"] = config.get("task_type", "CAUSAL_LM")
    config["modules_to_save"] = None
    config["fan_in_fan_out"] = False
    config["bias"] = "none"
    config["init_lora_weights"] = True
    config["peft_type"] = "LORA"
    config["use_dora"] = config.get("use_dora", False)
    config["use_rslora"] = config.get("use_rslora", False)
    config["lora_dropout"] = float(config.get("lora_dropout", 0.0))

    if candidate_spec["drop_lm_head"]:
        config["target_modules"] = [
            module for module in config["target_modules"] if module != "lm_head"
        ]

    return config


def svd_from_factors(
    lora_B: torch.Tensor,
    lora_A: torch.Tensor,
) -> tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    q_b, r_b = torch.linalg.qr(lora_B.float())
    q_a, r_a = torch.linalg.qr(lora_A.float().T)
    core = r_b @ r_a.T
    u_core, s, vh_core = torch.linalg.svd(core, full_matrices=False)
    u = q_b @ u_core
    vh = vh_core @ q_a.T
    return u, s, vh


def compress_dense_svd(
    lora_B: torch.Tensor,
    lora_A: torch.Tensor,
    rank: int,
) -> tuple[torch.Tensor, torch.Tensor, dict]:
    u, s, vh = svd_from_factors(lora_B, lora_A)
    k = min(rank, s.shape[0])
    sqrt_s = torch.sqrt(s[:k])
    new_B = (u[:, :k] * sqrt_s.unsqueeze(0)).to(lora_B.dtype).contiguous()
    new_A = (sqrt_s.unsqueeze(1) * vh[:k, :]).to(lora_A.dtype).contiguous()
    retained_energy = float(s[:k].square().sum() / s.square().sum()) if s.numel() else 1.0
    retained_mass = float(s[:k].sum() / s.sum()) if s.numel() else 1.0
    return new_B, new_A, {
        "rank": int(k),
        "retained_energy": retained_energy,
        "retained_singular_mass": retained_mass,
    }


def allocate_block_ranks(
    singular_values_by_component: list,
    total_rank: int,
    min_rank_per_component: int,
    component_priority: dict,
) -> dict:
    ranks = {}
    cursors = {}

    for name, singular_values in singular_values_by_component:
        guaranteed = min(min_rank_per_component, singular_values.numel())
        ranks[name] = guaranteed
        cursors[name] = guaranteed

    remaining = total_rank - sum(ranks.values())
    if remaining < 0:
        raise ValueError(
            f"Rank budget {total_rank} is smaller than guaranteed floor "
            f"{min_rank_per_component} x {len(singular_values_by_component)}"
        )

    while remaining > 0:
        best_name = None
        best_value = None
        for name, singular_values in singular_values_by_component:
            cursor = cursors[name]
            if cursor >= singular_values.numel():
                continue
            priority = component_priority.get(name, 1.0)
            value = float(singular_values[cursor]) * priority
            if best_value is None or value > best_value:
                best_name = name
                best_value = value

        if best_name is None:
            break

        ranks[best_name] += 1
        cursors[best_name] += 1
        remaining -= 1

    return ranks


def compress_block_topk(
    components: list,
    total_rank: int,
    min_rank_per_component: int,
    component_priority: dict,
) -> tuple[torch.Tensor, torch.Tensor, dict]:
    component_svds = []
    row_offset = 0

    for comp_name, lora_A, lora_B in components:
        u, s, vh = svd_from_factors(lora_B, lora_A)
        component_svds.append(
            {
                "name": comp_name,
                "u": u,
                "s": s,
                "vh": vh,
                "lora_A_dtype": lora_A.dtype,
                "lora_B_dtype": lora_B.dtype,
                "row_start": row_offset,
                "row_end": row_offset + lora_B.shape[0],
            }
        )
        row_offset += lora_B.shape[0]

    rank_allocation = allocate_block_ranks(
        [(item["name"], item["s"]) for item in component_svds],
        total_rank=total_rank,
        min_rank_per_component=min_rank_per_component,
        component_priority=component_priority,
    )

    fused_out_dim = row_offset
    merged_B = torch.zeros(fused_out_dim, total_rank, dtype=torch.float32)
    merged_A_parts = []
    rank_offset = 0
    component_stats = []
    total_energy = 0.0
    kept_energy = 0.0

    for item in component_svds:
        rank = rank_allocation[item["name"]]
        singular_values = item["s"]
        comp_total_energy = float(singular_values.square().sum()) if singular_values.numel() else 0.0
        comp_kept_energy = float(singular_values[:rank].square().sum()) if rank else 0.0
        total_energy += comp_total_energy
        kept_energy += comp_kept_energy

        if rank > 0:
            sqrt_s = torch.sqrt(singular_values[:rank])
            comp_B = item["u"][:, :rank] * sqrt_s.unsqueeze(0)
            comp_A = sqrt_s.unsqueeze(1) * item["vh"][:rank, :]
            merged_B[
                item["row_start"] : item["row_end"],
                rank_offset : rank_offset + rank,
            ] = comp_B
            merged_A_parts.append(comp_A)
            rank_offset += rank

        component_stats.append(
            {
                "component": item["name"],
                "allocated_rank": int(rank),
                "available_rank": int(singular_values.numel()),
                "retained_energy": (
                    comp_kept_energy / comp_total_energy if comp_total_energy else 1.0
                ),
            }
        )

    if merged_A_parts:
        merged_A = torch.cat(merged_A_parts, dim=0).to(
            components[0][1].dtype
        ).contiguous()
    else:
        merged_A = torch.zeros(
            0,
            components[0][1].shape[1],
            dtype=components[0][1].dtype,
        )

    merged_B = merged_B[:, :rank_offset].to(components[0][2].dtype).contiguous()
    return merged_B, merged_A, {
        "rank": int(rank_offset),
        "retained_energy": kept_energy / total_energy if total_energy else 1.0,
        "min_rank_per_component": int(min_rank_per_component),
        "component_priority": component_priority,
        "component_stats": component_stats,
    }


def build_output_tensors(candidate_spec: dict) -> tuple:
    output_tensors = {}
    fused_stats = []

    for base_name in source_base_names:
        lora_A = source_adapter_tensors[f"{base_name}.lora_A.weight"]
        lora_B = source_adapter_tensors[f"{base_name}.lora_B.weight"]

        if base_name in mamba_merge_bases:
            continue

        if ".experts.w3" in base_name and lora_A.numel() == 0:
            continue

        if candidate_spec["drop_lm_head"] and ".lm_head" in base_name:
            continue

        if ".experts.w1" in base_name or ".experts.w2" in base_name:
            if lora_A.shape[0] == 1:
                lora_A = lora_A.expand(lora_B.shape[0], -1, -1).contiguous()
            elif lora_B.shape[0] == 1:
                lora_B = lora_B.expand(lora_A.shape[0], -1, -1).contiguous()

            num_experts = lora_A.shape[0]
            for expert_idx in range(num_experts):
                expert_base = rewrite_expert_base_name(base_name, expert_idx)
                output_base = rename_base_name(
                    expert_base,
                    candidate_spec["namespace_mode"],
                )
                output_tensors[f"{output_base}.lora_A.weight"] = lora_A[expert_idx].contiguous()
                output_tensors[f"{output_base}.lora_B.weight"] = lora_B[expert_idx].contiguous()
            continue

        output_base = rename_base_name(
            base_name,
            candidate_spec["namespace_mode"],
        )
        output_tensors[f"{output_base}.lora_A.weight"] = lora_A.contiguous()
        output_tensors[f"{output_base}.lora_B.weight"] = lora_B.contiguous()

    for layer_prefix, components_map in sorted(mamba_merge_layers.items()):
        ordered_components = []
        for comp_name in ("gate_proj", "x_proj"):
            comp_base = components_map[comp_name]
            ordered_components.append(
                (
                    comp_name,
                    source_adapter_tensors[f"{comp_base}.lora_A.weight"],
                    source_adapter_tensors[f"{comp_base}.lora_B.weight"],
                )
            )

        if candidate_spec["strategy"] == "dense_svd":
            lora_A_parts = [item[1] for item in ordered_components]
            merged_lora_A = torch.cat(lora_A_parts, dim=0)

            row_offset = 0
            merged_rank = merged_lora_A.shape[0]
            fused_out_dim = sum(item[2].shape[0] for item in ordered_components)
            merged_lora_B = torch.zeros(
                fused_out_dim,
                merged_rank,
                dtype=merged_lora_A.dtype,
            )

            rank_offset = 0
            for _, _, lora_B in ordered_components:
                out_dim = lora_B.shape[0]
                rank = lora_B.shape[1]
                merged_lora_B[row_offset : row_offset + out_dim, rank_offset : rank_offset + rank] = lora_B
                row_offset += out_dim
                rank_offset += rank

            if merged_rank > FORCED_FUSED_RANK:
                merged_lora_B, merged_lora_A, stats = compress_dense_svd(
                    merged_lora_B,
                    merged_lora_A,
                    FORCED_FUSED_RANK,
                )
            else:
                stats = {
                    "rank": int(merged_rank),
                    "retained_energy": 1.0,
                    "retained_singular_mass": 1.0,
                }
        elif candidate_spec["strategy"] == "block_topk":
            merged_lora_B, merged_lora_A, stats = compress_block_topk(
                ordered_components,
                total_rank=FORCED_FUSED_RANK,
                min_rank_per_component=candidate_spec["min_rank_per_component"],
                component_priority=candidate_spec.get("component_priority", {}),
            )
        else:
            raise ValueError(f"Unknown strategy: {candidate_spec['strategy']}")

        output_base = rename_base_name(
            f"{layer_prefix}.in_proj",
            candidate_spec["namespace_mode"],
        )
        output_tensors[f"{output_base}.lora_A.weight"] = merged_lora_A
        output_tensors[f"{output_base}.lora_B.weight"] = merged_lora_B
        fused_stats.append(
            {
                "layer_prefix": layer_prefix,
                "strategy": candidate_spec["strategy"],
                **stats,
            }
        )

    return output_tensors, fused_stats


def save_candidate(candidate_spec: dict) -> dict:
    output_dir = OUTPUT_ROOT / candidate_spec["name"]
    if output_dir.is_dir():
        shutil.rmtree(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    output_tensors, fused_stats = build_output_tensors(candidate_spec)
    adapter_config = build_adapter_config(output_tensors, candidate_spec)

    save_file(output_tensors, output_dir / "adapter_model.safetensors")
    with open(output_dir / "adapter_config.json", "w") as f:
        json.dump(adapter_config, f, indent=2)
        f.write("\n")

    mean_retained_energy = (
        sum(item["retained_energy"] for item in fused_stats) / len(fused_stats)
        if fused_stats
        else 1.0
    )
    manifest = {
        **candidate_spec,
        "output_dir": str(output_dir),
        "tensor_count": len(output_tensors),
        "target_modules": adapter_config["target_modules"],
        "mean_fused_retained_energy": mean_retained_energy,
        "num_fused_layers": len(fused_stats),
    }

    with open(output_dir / "conversion_manifest.json", "w") as f:
        json.dump(
            {
                "manifest": manifest,
                "fused_layers": fused_stats,
            },
            f,
            indent=2,
        )
        f.write("\n")

    zip_base = WORK_DIR / f"submission_{candidate_spec['name']}"
    zip_path = Path(shutil.make_archive(str(zip_base), "zip", output_dir))
    manifest["zip_path"] = str(zip_path)
    return manifest


def _safe_merge_pool(manifests: list) -> list:
    pool = [
        m
        for m in manifests
        if m.get("namespace_mode") == "model" and not m.get("drop_lm_head", False)
    ]
    return pool if pool else manifests


def _pick_best_manifest(manifests: list) -> dict:
    pool = _safe_merge_pool(manifests)
    best_energy = max(float(m.get("mean_fused_retained_energy", 0.0)) for m in pool)
    tied = [
        m
        for m in pool
        if abs(float(m.get("mean_fused_retained_energy", 0.0)) - best_energy) < 1e-9
    ]
    for m in tied:
        if m.get("name") == ACTIVE_CANDIDATE_NAME:
            return m
    return max(
        pool,
        key=lambda m: (
            float(m.get("mean_fused_retained_energy", 0.0)),
            int(m.get("tensor_count", 0)),
        ),
    )


def main():
    if MERGE_SELECTION not in (
        "single",
        "named",
        "best_fused_energy",
        "best_shortlist",
    ):
        raise ValueError(MERGE_SELECTION)

    if MERGE_SELECTION == "single":
        selected_specs = [ACTIVE_CANDIDATE]
    elif MERGE_SELECTION == "best_shortlist":
        selected_specs = [CANDIDATE_BY_NAME[n] for n in MERGE_SHORTLIST]
    else:
        selected_specs = CANDIDATE_SPECS

    candidate_manifests = []
    for candidate_spec in selected_specs:
        candidate_manifests.append(save_candidate(candidate_spec))

    if MERGE_SELECTION == "named":
        selected_manifest = next(
            item for item in candidate_manifests if item["name"] == ACTIVE_CANDIDATE_NAME
        )
    elif MERGE_SELECTION in ("best_fused_energy", "best_shortlist"):
        selected_manifest = _pick_best_manifest(candidate_manifests)
    else:
        selected_manifest = candidate_manifests[0]

    compare_rows = sorted(
        [
            {
                "name": m["name"],
                "mean_fused_retained_energy": m.get("mean_fused_retained_energy"),
                "tensor_count": m.get("tensor_count"),
                "num_fused_layers": m.get("num_fused_layers"),
            }
            for m in candidate_manifests
        ],
        key=lambda row: (
            float(row["mean_fused_retained_energy"] or 0.0),
            int(row["tensor_count"] or 0),
        ),
        reverse=True,
    )
    with open(WORK_DIR / "merge_compare.json", "w") as f:
        json.dump(
            {
                "source_adapter_path": str(SOURCE_ADAPTER_PATH),
                "merge_selection": MERGE_SELECTION,
                "selected": selected_manifest["name"],
                "candidates": compare_rows,
            },
            f,
            indent=2,
        )
        f.write("\n")

    submission_zip = WORK_DIR / "submission.zip"
    shutil.copy2(selected_manifest["zip_path"], submission_zip)

    sel_record = dict(selected_manifest)
    sel_record["merge_selection"] = MERGE_SELECTION
    sel_record["source_adapter_path"] = str(SOURCE_ADAPTER_PATH)
    with open(WORK_DIR / "selected_candidate.json", "w") as f:
        json.dump(sel_record, f, indent=2)
        f.write("\n")

    if ENABLE_DPO:
        run_dpo_from_rollouts(selected_manifest, submission_zip)
    elif ENABLE_SHORT_SFT:
        run_short_sft(selected_manifest, submission_zip)


main()


In [ ]:
# =====================================================================
# Direction 1 - Post-conversion fix: faithful LoRA scaling + rank safety
# ---------------------------------------------------------------------
# The conversion path rebuilds raw  B @ A  and writes adapter_config with
# r = lora_alpha = 32  (=> loader scaling 1.0). If the SOURCE adapter was
# trained with lora_alpha != r (or rslora), the converted adapter applies
# the LoRA delta at the WRONG magnitude. This cell:
#   1. prints what the source adapter actually specified (diagnostic),
#   2. frees disk left over from the conversion step,
#   3. folds the true source scaling into the lora_B tensors,
#   4. zero-pads every module to rank 32 so all modules share one rank
#      (loader-agnostic; zero rows/cols leave the delta B@A unchanged),
#   5. re-packages submission.zip from the corrected adapter.
#
# NOTE: run 66 confirmed source adapter v20 has r == lora_alpha == 32 and
# all 6005 modules already at rank 32 -> the fold/pad is a verified no-op.
# APPLY_SCALING_FIX is left False to skip the slow ~3 GB re-zip. Flip it
# True only if you swap to a source adapter with different r / lora_alpha.
# The diagnostic + disk cleanup below always run regardless of the flag.
# =====================================================================
APPLY_SCALING_FIX = False

import json
import math
import re
import shutil
from pathlib import Path

import torch
from safetensors import safe_open
from safetensors.torch import save_file

# ---- diagnostic: what does the SOURCE adapter actually specify? ----
_src_r = source_adapter_config.get("r")
_src_alpha = source_adapter_config.get("lora_alpha")
_src_rslora = bool(source_adapter_config.get("use_rslora", False))
_src_dora = bool(source_adapter_config.get("use_dora", False))
_src_rp = source_adapter_config.get("rank_pattern") or {}
_src_ap = source_adapter_config.get("alpha_pattern") or {}
if _src_r and _src_alpha:
    _src_scaling = _src_alpha / (math.sqrt(_src_r) if _src_rslora else _src_r)
else:
    _src_scaling = None

print("=== SOURCE adapter config ===")
print(f"  r={_src_r}  lora_alpha={_src_alpha}  use_rslora={_src_rslora}  use_dora={_src_dora}")
print(f"  effective source scaling = {_src_scaling}")
print(f"  source rank_pattern entries={len(_src_rp)}  alpha_pattern entries={len(_src_ap)}")
if _src_rp or _src_ap:
    print("  NOTE: source uses per-module rank/alpha patterns -> the uniform-scaling")
    print("        assumption may be approximate; review before trusting the fix.")

# ---- locate the adapter main() selected & zipped ----
with open(WORK_DIR / "selected_candidate.json") as _f:
    _selected = json.load(_f)
_adapter_dir = Path(_selected["output_dir"])
_cfg_path = _adapter_dir / "adapter_config.json"
_wts_path = _adapter_dir / "adapter_model.safetensors"
print(f"Selected converted adapter: {_adapter_dir}")

# ---- free disk: drop conversion leftovers not needed for submission ----
for _z in sorted(WORK_DIR.glob("submission_*.zip")):
    try:
        _z.unlink()
        print(f"  freed {_z.name}")
    except OSError:
        pass
_cand_root = _adapter_dir.parent
if _cand_root.name == "adapter_candidates":
    for _d in sorted(_cand_root.iterdir()):
        if _d.is_dir() and _d != _adapter_dir:
            shutil.rmtree(_d, ignore_errors=True)
            print(f"  freed candidate dir {_d.name}")

with open(_cfg_path) as _f:
    _cfg = json.load(_f)
_tensors = {}
with safe_open(_wts_path, framework="pt", device="cpu") as _f:
    for _k in _f.keys():
        _tensors[_k] = _f.get_tensor(_k)

_bases = sorted({re.sub(r"\.lora_[AB]\.weight$", "", k) for k in _tensors})
_bases = [
    b for b in _bases
    if f"{b}.lora_A.weight" in _tensors and f"{b}.lora_B.weight" in _tensors
]
_ranks = {b: int(_tensors[f"{b}.lora_A.weight"].shape[0]) for b in _bases}
_rank_hist = {}
for _r in _ranks.values():
    _rank_hist[_r] = _rank_hist.get(_r, 0) + 1

print("=== CONVERTED adapter ===")
print(f"  modules={len(_bases)}  rank histogram={dict(sorted(_rank_hist.items()))}")
print(f"  config r={_cfg.get('r')}  lora_alpha={_cfg.get('lora_alpha')}")
_mixed = len(_rank_hist) > 1
_over = [b for b, r in _ranks.items() if r > FORCED_FUSED_RANK]

# ---- decide / apply ----
if not APPLY_SCALING_FIX:
    print("APPLY_SCALING_FIX=False -> converted adapter left unchanged "
          "(submission.zip stays as the conversion output).")
elif _src_dora:
    print("ABORT FIX: source adapter uses DoRA; folding B@A scaling is invalid for DoRA.")
elif _src_scaling is None:
    print("ABORT FIX: source r/lora_alpha unavailable; cannot compute scaling.")
elif _over:
    print(f"ABORT FIX: {len(_over)} module(s) exceed rank {FORCED_FUSED_RANK} "
          f"(e.g. {_over[0]} r={_ranks[_over[0]]}); needs re-compression, not padding.")
else:
    _fixed = {}
    for _b in _bases:
        _a = _tensors[f"{_b}.lora_A.weight"]
        _bw = _tensors[f"{_b}.lora_B.weight"]
        _r = _a.shape[0]
        # 1) fold true source scaling into B  ->  (s*B) @ A == s*(B@A)
        _bw = (_bw.float() * _src_scaling).to(_bw.dtype)
        # 2) zero-pad up to FORCED_FUSED_RANK (delta unchanged by zero rows/cols)
        _pad = FORCED_FUSED_RANK - _r
        if _pad > 0:
            _a = torch.cat([_a, torch.zeros(_pad, _a.shape[1], dtype=_a.dtype)], dim=0)
            _bw = torch.cat([_bw, torch.zeros(_bw.shape[0], _pad, dtype=_bw.dtype)], dim=1)
        _fixed[f"{_b}.lora_A.weight"] = _a.contiguous()
        _fixed[f"{_b}.lora_B.weight"] = _bw.contiguous()

    # 3) config: scaling baked in, all modules rank 32 -> loader scaling = 32/32 = 1.0
    _cfg["r"] = FORCED_FUSED_RANK
    _cfg["lora_alpha"] = FORCED_FUSED_RANK
    _cfg["rank_pattern"] = {}
    _cfg["alpha_pattern"] = {}
    _cfg["use_rslora"] = False
    _cfg["use_dora"] = False

    save_file(_fixed, _wts_path)
    with open(_cfg_path, "w") as _f:
        json.dump(_cfg, _f, indent=2)
        _f.write("\n")

    # zip straight to submission.zip (no intermediate copy -> saves disk)
    shutil.make_archive(str(WORK_DIR / "submission"), "zip", _adapter_dir)

    print("=== FIX APPLIED ===")
    if abs(_src_scaling - 1.0) > 1e-6:
        print(f"  * source scaling {_src_scaling:.4f} != 1.0 -> magnitude WAS wrong; "
              f"folded into {len(_bases)} modules.")
    else:
        print("  * source scaling == 1.0 -> magnitude was already correct (no-op fold).")
    if _mixed:
        print(f"  * mixed ranks {sorted(_rank_hist)} -> zero-padded all modules "
              f"to rank {FORCED_FUSED_RANK}.")
    else:
        print(f"  * all modules already rank {FORCED_FUSED_RANK}.")
    print(f"  submission.zip re-packaged "
          f"({(WORK_DIR / 'submission.zip').stat().st_size / 1024 / 1024:.2f} MB)")


In [ ]:
# =====================================================================
# Direction 3 - Solver-distillation SFT (+ bit_8) + Rationalization-STaR
# ---------------------------------------------------------------------
# Phase A: programmatic solvers (instant, CPU) for the deterministic
# families produce verified-correct traces:
#   numeral_convert : standard Roman numerals          (100%)
#   physics         : d = 0.5*g*t^2, recover g         (100% within tol)
#   unit_conv       : output = k * input, recover k    (100% within tol)
#   text_crypto     : letter-substitution cipher        (~99.5%)
#   bit_8 (linear)  : structured bit-op search          (~43%)
#
# Phase B: rationalization-STaR for the *hard* families (bit_8 unsolved +
# symbol_equation). For each hard row, hand the model the gold answer as
# a constraint and ask it to construct a clean step-by-step derivation.
# Knowing both the examples AND the answer dramatically narrows the rule
# space -- the model often finds a coherent rule it couldn't reach
# forward. Each trace is verified against gold and filtered for leakage,
# then paired with the *bare* prompt for SFT. This is the half of STaR
# explicitly designed for breaking model ceilings.
#
# All in one Kaggle run, no external data. Set the two flags below to
# disable either phase.
# =====================================================================
ENABLE_SOLVER_SFT    = True

# Tightened for Kaggle's 12h hard cap. The prior 7k-trace x 2-epoch SFT
# alone took ~11h on this base model; 1 epoch + a smaller rationalize
# phase + a hard watchdog keeps the whole run under 11h with margin.
SSFT_MAX_SEQ_LEN     = 768
SSFT_EPOCHS          = 1.0     # was 2.0; one epoch over solver + rationalize fits budget
SSFT_LR              = 5e-5
SSFT_GRAD_ACCUM      = 8
SSFT_WARMUP_RATIO    = 0.05
SSFT_ENABLE_THINKING = False   # keep SFT chat template consistent with eval
SSFT_SEED            = 42

# ---- Phase B: rationalization for hard families ----
RATION_ENABLED         = True
RATION_MAX_ROWS        = 500   # cap rows (~2-3h on RTX Pro 6000)
RATION_TIME_BUDGET_MIN = 180   # HARD STOP: bail out after this many minutes
RATION_MAX_NEW_TOKENS  = 512
RATION_TEMP            = 0.8
RATION_TOP_P           = 0.95
RATION_MIN_CHARS       = 80    # drop trivially short traces
RATION_SEED            = 42

NEMOTRON_UTIL_DIR = "/kaggle/usr/lib/notebooks/ryanholbrook/nvidia-utility-script/"

RATION_TEMPLATE = (
    "{prompt}\n\n"
    "The correct final answer is {gold}. Produce a clean step-by-step solution "
    "that derives this answer from the examples -- figure out the rule that fits "
    "every example AND yields this answer for the query. Do not mention that the "
    "answer was given; reason as if solving from scratch. End with \\boxed{{{gold}}}."
)
_RATION_LEAK_PAT = re.compile(
    r"(?i)(answer (was|is) (given|provided|told)|"
    r"the (correct )?(final )?answer (was|is) (already |)?(provided|given|told)|"
    r"you (gave|told|provided) (me )?the (correct )?answer|"
    r"as (the )?(hint|prompt|instructions?) (said|state[ds]?)|"
    r"\bgiven (the |that |you (gave|provided|told|said) )?(the answer|the correct|the result))"
)
_RATION_THINK_PAT = re.compile(r"<think>.*?</think>\s*", re.S)

_ROMAN = [(1000, 'M'), (900, 'CM'), (500, 'D'), (400, 'CD'), (100, 'C'),
          (90, 'XC'), (50, 'L'), (40, 'XL'), (10, 'X'), (9, 'IX'),
          (5, 'V'), (4, 'IV'), (1, 'I')]


def _ssft_family(prompt: str) -> str:
    s = str(prompt)[:1200].lower()
    if '8-bit binary' in s or ('bit manipulation' in s and 'binary' in s):
        return 'bit_8'
    if 'secret encryption' in s or ('decrypt' in s and 'wonderland' in s):
        return 'text_crypto'
    if 'numeral system' in s:
        return 'numeral_convert'
    if 'unit conversion' in s:
        return 'unit_conv'
    if 'gravitational' in s or 'falling distance' in s:
        return 'physics'
    if 'equation' in s and 'transformation' in s:
        return 'symbol_equation'
    return 'other'


def _to_roman(n: int) -> str:
    r = ''
    for v, s in _ROMAN:
        while n >= v:
            r += s
            n -= v
    return r


def _trace_numeral(p, gold):
    import re
    m = re.search(r'write the number (\d+) in', p)
    if not m:
        return None
    n = int(m.group(1))
    if _to_roman(n) != gold:
        return None
    steps = [f"{v} = {_to_roman(v)}" for v in
             (n // 1000 * 1000, n % 1000 // 100 * 100, n % 100 // 10 * 10, n % 10) if v]
    return ("The examples follow standard Roman numerals. Break the number into "
            "place values and convert each: " + "; ".join(steps) +
            f". Joining these gives {gold}.\n\\boxed{{{gold}}}")


def _trace_physics(p, gold):
    import re
    ex = re.findall(r't = ([\d.]+)\s*s,\s*distance = ([\d.]+)\s*m', p)
    q = re.search(r'for t = ([\d.]+)\s*s given', p)
    if not ex or not q:
        return None
    lo, hi = -1e9, 1e9
    for t, d in ex:
        t = float(t); d = float(d)
        lo = max(lo, 2 * (d - 0.005) / t ** 2)
        hi = min(hi, 2 * (d + 0.005) / t ** 2)
    g = (lo + hi) / 2
    tq = float(q.group(1))
    if abs(0.5 * g * tq ** 2 - float(gold)) > 0.05:
        return None
    t0, d0 = ex[0]
    return ("The relationship is d = 0.5*g*t^2, so g = 2*d/t^2 is constant. "
            f"From the example t = {t0}s, d = {d0}m: g = 2*{d0}/{t0}^2 ~= "
            f"{2 * float(d0) / float(t0) ** 2:.3f}. All examples agree on g ~= "
            f"{g:.3f}. For t = {tq}s: d = 0.5 * {g:.3f} * {tq}^2 ~= "
            f"{gold}.\n\\boxed{{{gold}}}")


def _trace_unit(p, gold):
    import re
    ex = re.findall(r'([\d.]+)\s*m becomes ([\d.]+)', p)
    q = re.search(r'convert the following measurement:\s*([\d.]+)\s*m', p)
    if not ex or not q:
        return None
    lo, hi = -1e9, 1e9
    for a, b in ex:
        a = float(a); b = float(b)
        lo = max(lo, (b - 0.005) / a)
        hi = min(hi, (b + 0.005) / a)
    k = (lo + hi) / 2
    x = float(q.group(1))
    if abs(x * k - float(gold)) > 0.05:
        return None
    a0, b0 = ex[0]
    return ("Each measurement is scaled by a fixed factor k = output / input. "
            f"From {a0} m -> {b0}: k = {b0}/{a0} ~= {float(b0) / float(a0):.4f}. "
            f"All examples agree on k ~= {k:.4f}. "
            f"For {x} m: {x} * {k:.4f} ~= {gold}.\n\\boxed{{{gold}}}")


def _trace_crypto(p, gold, vocab):
    import re
    pairs = re.findall(r'^(.+?) -> (.+?)\s*$', p, re.M)
    qm = re.search(r'decrypt the following text:\s*(.+)', p)
    if not qm:
        return None
    words = qm.group(1).strip().split()
    cmap = {}
    for ct, pt in pairs:
        cw, pw = ct.strip().split(), pt.strip().split()
        if len(cw) != len(pw):
            continue
        for a, b in zip(cw, pw):
            if len(a) == len(b):
                for x, y in zip(a, b):
                    cmap[x] = y
    for _ in range(12):
        changed = False
        for w in words:
            if '?' not in ''.join(cmap.get(c, '?') for c in w):
                continue
            cands = []
            for v in vocab:
                if len(v) != len(w):
                    continue
                ok = True
                trial = {}
                for c, pc in zip(w, v):
                    if (c in cmap and cmap[c] != pc) or (c in trial and trial[c] != pc):
                        ok = False
                        break
                    trial[c] = pc
                if ok:
                    cands.append(v)
            if len(cands) == 1:
                for c, pc in zip(w, cands[0]):
                    if c not in cmap:
                        cmap[c] = pc
                        changed = True
        if not changed:
            break
    decoded = [''.join(cmap.get(c, '?') for c in w) for w in words]
    if ' '.join(decoded) != gold:
        return None
    deriv = None
    for ct, pt in pairs:
        cw, pw = ct.strip().split(), pt.strip().split()
        if len(cw) == len(pw):
            for a, b in zip(cw, pw):
                if len(a) == len(b) and len(a) >= 3:
                    deriv = (a, b)
                    break
        if deriv:
            break
    used = {c: cmap[c] for w in words for c in w if c in cmap}
    mapping = ", ".join(f"{c}->{p2}" for c, p2 in sorted(used.items()))
    pairlines = "; ".join(f"{cw} -> {dw}" for cw, dw in zip(words, decoded))
    d = ""
    if deriv:
        d = (f" For example, '{deriv[0]} -> {deriv[1]}' aligns as "
             + ", ".join(f"{x}->{y}" for x, y in zip(*deriv)) + ".")
    return ("This is a letter-substitution cipher: align each ciphertext word "
            "with its plaintext letter by letter to recover the mapping." + d +
            f" Collecting the mappings needed for the target: {mapping}. "
            f"Decoding word by word: {pairlines}.\n\\boxed{{{gold}}}")


# ----------------------- bit_8 structured solver -----------------------
def _rotl(x, k): return ((x << k) | (x >> (8 - k))) & 0xff
def _rotr(x, k): return ((x >> k) | (x << (8 - k))) & 0xff

_BIT_PRIM = {
    'id':   (lambda x: x, 'leave the bits unchanged'),
    'not':  (lambda x: (~x) & 0xff, 'invert every bit'),
    'rev':  (lambda x: int(f'{x:08b}'[::-1], 2), 'reverse the bit order'),
    'gray': (lambda x: x ^ (x >> 1), 'XOR each bit with the bit to its right'),
    'swap': (lambda x: ((x << 4) | (x >> 4)) & 0xff, 'swap the two 4-bit halves'),
}
for _k in range(1, 8):
    _BIT_PRIM[f'rotl{_k}'] = ((lambda k: lambda x: _rotl(x, k))(_k), f'rotate left by {_k}')
    _BIT_PRIM[f'rotr{_k}'] = ((lambda k: lambda x: _rotr(x, k))(_k), f'rotate right by {_k}')
    _BIT_PRIM[f'shl{_k}'] = ((lambda k: lambda x: (x << k) & 0xff)(_k), f'shift left by {_k}')
    _BIT_PRIM[f'shr{_k}'] = ((lambda k: lambda x: x >> k)(_k), f'shift right by {_k}')

_BIT_HYP = dict(_BIT_PRIM)
for _n1, (_f1, _d1) in list(_BIT_PRIM.items()):
    for _n2, (_f2, _d2) in list(_BIT_PRIM.items()):
        _BIT_HYP[f'{_n2}o{_n1}'] = (
            (lambda a, b: lambda x: b(a(x)))(_f1, _f2), f'{_d1}, then {_d2}')
        if _n1 < _n2:
            _BIT_HYP[f'{_n1}^{_n2}'] = (
                (lambda a, b: lambda x: a(x) ^ b(x))(_f1, _f2),
                f'({_d1}) XORed with ({_d2})')


def _ca_rule(r):
    def f(x):
        b = [(x >> i) & 1 for i in range(8)]
        o = 0
        for j in range(8):
            nb = (b[(j + 1) % 8] << 2) | (b[j] << 1) | b[(j - 1) % 8]
            if (r >> nb) & 1:
                o |= (1 << j)
        return o
    return f


for _r in range(256):
    _BIT_HYP[f'ca{_r}'] = (_ca_rule(_r),
                           'replace each bit by a fixed function of it and its two neighbours')


def _mask_and(pairs):
    m = 0
    for p in range(8):
        one = any((o >> p) & 1 for i, o in pairs)
        zero_in = any(((i >> p) & 1) and not ((o >> p) & 1) for i, o in pairs)
        seen = any((i >> p) & 1 for i, o in pairs)
        if one and zero_in:
            return None
        if one:
            m |= (1 << p)
        elif not seen:
            return None
    return m if all((i & m) == o for i, o in pairs) else None


def _mask_or(pairs):
    m = 0
    for p in range(8):
        one = any(not ((i >> p) & 1) and ((o >> p) & 1) for i, o in pairs)
        zero = any(not ((i >> p) & 1) and not ((o >> p) & 1) for i, o in pairs)
        seen = any(not ((i >> p) & 1) for i, o in pairs)
        if one and zero:
            return None
        if one:
            m |= (1 << p)
        elif not seen:
            return None
    return m if all((i | m) == o for i, o in pairs) else None


def _solve_bit8(prompt):
    import re
    ex = re.findall(r'([01]{8})\s*->\s*([01]{8})', prompt)
    q = re.search(r'determine the output for:\s*([01]{8})', prompt)
    if not ex or not q:
        return None
    pairs = [(int(i, 2), int(o, 2)) for i, o in ex]
    qin = int(q.group(1), 2)
    preds = {}
    for name, (g, desc) in _BIT_HYP.items():
        if all(g(i) == o for i, o in pairs):
            preds.setdefault(g(qin), desc)
        else:
            c = g(pairs[0][0]) ^ pairs[0][1]
            if all((g(i) ^ c) == o for i, o in pairs):
                d = desc if c == 0 else f'{desc}, then XOR with mask {c:08b}'
                preds.setdefault(g(qin) ^ c, d)
    am = _mask_and(pairs)
    if am is not None:
        preds.setdefault(qin & am, f'keep only the bits in mask {am:08b} (bitwise AND)')
    om = _mask_or(pairs)
    if om is not None:
        preds.setdefault(qin | om, f'set the bits in mask {om:08b} (bitwise OR)')
    if len(preds) == 1:
        v, desc = next(iter(preds.items()))
        return f'{v:08b}', desc
    return None


def _trace_bit8(p, gold):
    import re
    r = _solve_bit8(p)
    if r is None or r[0] != gold:
        return None
    ex = re.findall(r'([01]{8})\s*->\s*([01]{8})', p)
    q = re.search(r'determine the output for:\s*([01]{8})', p)
    chk = "; ".join(f'{i} -> {o}' for i, o in ex[:2])
    return ("This is a fixed bit transformation. Comparing the example inputs "
            "and outputs, the single rule that reproduces every example is: "
            f"{r[1]}. It checks out on the examples ({chk}). Applying the same "
            f"rule to {q.group(1)} gives {gold}.\n\\boxed{{{gold}}}")


def _resolve_packed_paths(util_dir: str) -> None:
    """Append .pth-referenced packed package dirs (cutlass, ...) to sys.path."""
    import sys

    base = Path(util_dir)
    if not base.is_dir():
        print(f"[SSFT] util dir not found: {util_dir}")
        return
    for pth in sorted(base.glob("*.pth")):
        try:
            rel = pth.read_text().strip()
        except OSError:
            continue
        if not rel:
            continue
        packed = pth.parent / rel
        if packed.exists() and str(packed) not in sys.path:
            sys.path.append(str(packed))
            print(f"[SSFT] sys.path += {packed}")


def _fix_triton_ptxas() -> None:
    """Copy the bundled Blackwell ptxas to a writable, executable location."""
    import os
    import shutil
    import stat

    util = NEMOTRON_UTIL_DIR.rstrip("/")
    src = f"{util}/triton/backends/nvidia/bin/ptxas-blackwell"
    dst = "/tmp/ptxas-blackwell"
    if not os.path.exists(src):
        print(f"[SSFT] ptxas-blackwell not found ({src}); skipping triton fix")
        return
    if os.path.isdir(dst):
        shutil.rmtree(dst, ignore_errors=True)
    shutil.copy2(src, dst)
    os.chmod(dst, os.stat(dst).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
    try:
        import triton.backends.nvidia as nv_backend

        src_bin = os.path.join(os.path.dirname(nv_backend.__file__), "bin")
        dst_bin = "/tmp/triton_nvidia_bin"
        shutil.copytree(src_bin, dst_bin, dirs_exist_ok=True)
        for f in os.listdir(dst_bin):
            fp = os.path.join(dst_bin, f)
            if os.path.isfile(fp):
                os.chmod(fp, os.stat(fp).st_mode | stat.S_IEXEC | stat.S_IXGRP | stat.S_IXOTH)
        nv_backend.__file__ = os.path.join(dst_bin, "..", "__init__.py")
    except Exception as e:
        print(f"[SSFT] triton bin copy skipped: {e}")
    os.environ["TRITON_PTXAS_PATH"] = dst
    os.environ["TRITON_PTXAS_BLACKWELL_PATH"] = dst
    try:
        import triton.backends.nvidia.compiler as nv_compiler

        nv_compiler.get_ptxas_version = lambda arch: "12.0"
    except Exception:
        pass
    print("[SSFT] triton ptxas-blackwell fix applied")


def _build_solver_traces():
    """Generate verified reasoning traces for all solvable families."""
    import re
    import pandas as pd

    train_path = _resolve_train_csv()
    df = pd.read_csv(train_path).dropna(subset=["prompt", "answer"])
    df["prompt"] = df["prompt"].astype(str)
    df["answer"] = df["answer"].astype(str).str.strip()
    df["family"] = df["prompt"].map(_ssft_family)

    vocab = set()
    for p in df[df["family"] == "text_crypto"]["prompt"]:
        for _ct, pt in re.findall(r'^(.+?) -> (.+?)\s*$', p, re.M):
            for w in pt.strip().split():
                if w.isalpha():
                    vocab.add(w)

    rows = []
    per_fam = {}
    for _, r in df.iterrows():
        fam = r["family"]
        p, gold = r["prompt"], r["answer"]
        try:
            if fam == "numeral_convert":
                tr = _trace_numeral(p, gold)
            elif fam == "physics":
                tr = _trace_physics(p, gold)
            elif fam == "unit_conv":
                tr = _trace_unit(p, gold)
            elif fam == "text_crypto":
                tr = _trace_crypto(p, gold, vocab)
            elif fam == "bit_8":
                tr = _trace_bit8(p, gold)
            else:
                tr = None
        except Exception:
            tr = None
        if tr is not None:
            rows.append({"user": p.strip(), "assistant": tr})
            per_fam[fam] = per_fam.get(fam, 0) + 1
    return rows, per_fam


def run_solver_sft(adapter_dir, submission_zip) -> None:
    import gc
    import os
    import random
    import sys

    if not torch.cuda.is_available():
        raise RuntimeError("ENABLE_SOLVER_SFT needs a GPU kernel")
    _clear_merge_weights()

    for _z in WORK_DIR.glob("submission_*.zip"):
        try:
            _z.unlink()
        except OSError:
            pass

    _resolve_packed_paths(NEMOTRON_UTIL_DIR)
    try:
        import cutlass  # noqa: F401
        print("[SSFT] cutlass import OK")
    except ImportError as _e:
        print(f"[SSFT] WARNING: cutlass not importable ({_e}) -- model load may fail")
    _fix_triton_ptxas()

    from datasets import Dataset
    from peft import PeftModel
    from transformers import (
        AutoModelForCausalLM,
        AutoTokenizer,
        Trainer,
        TrainingArguments,
    )

    random.seed(SSFT_SEED)
    torch.manual_seed(SSFT_SEED)

    base_dir = _resolve_base_model_dir()
    adapter_dir = Path(adapter_dir)
    if not (adapter_dir / "adapter_config.json").is_file():
        raise FileNotFoundError(adapter_dir)

    # ---- generate verified solver traces (instant, CPU) ----
    traces, per_fam = _build_solver_traces()
    random.shuffle(traces)
    print(f"[SSFT] verified traces: {len(traces)} | per family: {per_fam}")
    for ex in traces[:3]:
        print("  --- sample ---")
        print("  USER :", ex["user"][:150].replace(chr(10), " "))
        print("  ASST :", ex["assistant"].replace(chr(10), " ")[:320])
    if len(traces) < 100:
        raise RuntimeError(f"Only {len(traces)} traces generated; check solvers.")

    # ---- tokenizer + base model + converted adapter ----
    tokenizer = AutoTokenizer.from_pretrained(
        str(base_dir), trust_remote_code=True, local_files_only=True
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    model = AutoModelForCausalLM.from_pretrained(
        str(base_dir),
        device_map={"": 0},
        trust_remote_code=True,
        dtype=torch.bfloat16,
        local_files_only=True,
    )
    for _name, _mod in list(sys.modules.items()):
        if "modeling_nemotron_h" in _name:
            _mod.is_fast_path_available = False
    model = PeftModel.from_pretrained(model, str(adapter_dir), is_trainable=True)

    # ---- Phase B: rationalization-STaR for hard families ----
    if RATION_ENABLED:
        import pandas as _pd
        train_path2 = _resolve_train_csv()
        df_all = _pd.read_csv(train_path2).dropna(subset=["prompt", "answer"])
        df_all["prompt"] = df_all["prompt"].astype(str)
        df_all["answer"] = df_all["answer"].astype(str).str.strip()
        df_all["family"] = df_all["prompt"].map(_ssft_family)
        done_users = {t["user"] for t in traces}
        hard_mask = df_all["family"].isin(["bit_8", "symbol_equation"])
        not_done = ~df_all["prompt"].str.strip().isin(done_users)
        hard = df_all[hard_mask & not_done]
        if len(hard) > RATION_MAX_ROWS:
            hard = hard.sample(n=RATION_MAX_ROWS, random_state=RATION_SEED)
        hard = hard.reset_index(drop=True)
        print(f"[RATION] hard rows to rationalize: {len(hard)} "
              f"(bit_8 unsolved + symbol_equation)")

        model.eval()
        device = next(model.parameters()).device

        def _norm_ans(s):
            return str(s).strip().lower().replace("$", "").replace(" ", "")

        def _last_boxed(text):
            ms = re.findall(r"\\boxed\{([^}]*)\}", text or "")
            return ms[-1].strip() if ms else None

        def _is_correct(text, gold):
            ext = _last_boxed(text)
            if ext is None:
                return False
            if _norm_ans(ext) == _norm_ans(gold):
                return True
            try:
                g = float(str(gold).replace(" ", ""))
                return abs(float(ext) - g) <= max(0.02, 1e-3 * abs(g))
            except ValueError:
                return False

        def _ration_encode(user_msg):
            ck = {"chat_template_kwargs": {"enable_thinking": False}}
            try:
                enc = tokenizer.apply_chat_template(
                    [{"role": "user", "content": user_msg}],
                    tokenize=True, add_generation_prompt=True,
                    return_dict=True, return_tensors="pt", **ck,
                )
            except TypeError:
                enc = tokenizer.apply_chat_template(
                    [{"role": "user", "content": user_msg}],
                    tokenize=True, add_generation_prompt=True,
                    return_dict=True, return_tensors="pt",
                )
            return {k: v.to(device) for k, v in enc.items()}

        kept_r = 0
        dropped_leak = dropped_wrong = dropped_short = call_failed = 0
        per_fam_r = {}
        import time as _time
        ration_t0 = _time.time()
        ration_deadline = ration_t0 + RATION_TIME_BUDGET_MIN * 60
        for i, row in hard.iterrows():
            if _time.time() > ration_deadline:
                print(f"  [RATION] time-budget watchdog tripped at row {i}/"
                      f"{len(hard)} (kept {kept_r}); stopping rationalization "
                      f"so SFT can finish within Kaggle's 12h cap.", flush=True)
                break
            p_text = row["prompt"]
            gold = row["answer"]
            fam = row["family"]
            user_msg = RATION_TEMPLATE.format(prompt=p_text, gold=gold)
            try:
                enc = _ration_encode(user_msg)
                with torch.inference_mode():
                    out = model.generate(
                        input_ids=enc["input_ids"],
                        attention_mask=enc["attention_mask"],
                        max_new_tokens=RATION_MAX_NEW_TOKENS,
                        do_sample=True,
                        temperature=RATION_TEMP,
                        top_p=RATION_TOP_P,
                        pad_token_id=tokenizer.pad_token_id,
                        eos_token_id=tokenizer.eos_token_id,
                    )
                gen_text = tokenizer.decode(
                    out[0, enc["input_ids"].shape[1]:],
                    skip_special_tokens=True,
                ).strip()
            except Exception as _ex:
                call_failed += 1
                continue
            clean = _RATION_THINK_PAT.sub("", gen_text).strip()
            # Trim to last \boxed{} inclusive (drops post-answer rambling).
            m = list(re.finditer(r"\\boxed\{[^}]*\}", clean))
            if m:
                clean = clean[: m[-1].end()]
            if not _is_correct(clean, gold):
                dropped_wrong += 1
                continue
            if _RATION_LEAK_PAT.search(clean):
                dropped_leak += 1
                continue
            if len(clean) < RATION_MIN_CHARS:
                dropped_short += 1
                continue
            # Pair the *bare* prompt with the rationalized trace.
            traces.append({"user": p_text.strip(), "assistant": clean})
            kept_r += 1
            per_fam_r[fam] = per_fam_r.get(fam, 0) + 1
            if (i + 1) % 25 == 0:
                rate = (i + 1) / max(1, _time.time() - ration_t0)
                eta = (len(hard) - i - 1) / max(rate, 1e-6) / 60
                print(f"  [RATION] {i+1}/{len(hard)} | kept {kept_r} "
                      f"| dropped wrong={dropped_wrong} leak={dropped_leak} "
                      f"short={dropped_short} call={call_failed} "
                      f"| {rate:.2f} rows/s ETA {eta:.1f}min", flush=True)
        print(f"[RATION] DONE kept {kept_r} | per family {per_fam_r} "
              f"| total traces now {len(traces)}")

    # ---- tokenize with assistant-only loss ----
    raw_ds = Dataset.from_list(traces)

    def tokenize_batch(examples):
        out_ids, out_mask, out_labels = [], [], []
        ck = {"chat_template_kwargs": {"enable_thinking": SSFT_ENABLE_THINKING}}
        for ut, at in zip(examples["user"], examples["assistant"]):
            msgs = [
                {"role": "user", "content": ut},
                {"role": "assistant", "content": at},
            ]
            try:
                full_text = tokenizer.apply_chat_template(
                    msgs, tokenize=False, add_generation_prompt=False, **ck
                )
                prefix_text = tokenizer.apply_chat_template(
                    [msgs[0]], tokenize=False, add_generation_prompt=True, **ck
                )
            except TypeError:
                full_text = tokenizer.apply_chat_template(
                    msgs, tokenize=False, add_generation_prompt=False
                )
                prefix_text = tokenizer.apply_chat_template(
                    [msgs[0]], tokenize=False, add_generation_prompt=True
                )
            full_enc = tokenizer(
                full_text, truncation=True, max_length=SSFT_MAX_SEQ_LEN, padding="max_length"
            )
            pref_enc = tokenizer(
                prefix_text, truncation=True, max_length=SSFT_MAX_SEQ_LEN, padding=False
            )
            lab = list(full_enc["input_ids"])
            pref_len = min(len(pref_enc["input_ids"]), len(lab))
            for i in range(pref_len):
                lab[i] = -100
            for i, m in enumerate(full_enc["attention_mask"]):
                if m == 0:
                    lab[i] = -100
            out_ids.append(full_enc["input_ids"])
            out_mask.append(full_enc["attention_mask"])
            out_labels.append(lab)
        return {"input_ids": out_ids, "attention_mask": out_mask, "labels": out_labels}

    tok_ds = raw_ds.map(tokenize_batch, batched=True, remove_columns=raw_ds.column_names)

    # ---- SFT the converted adapter on the verified traces ----
    model.gradient_checkpointing_enable()
    model.enable_input_require_grads()
    model.config.use_cache = False
    model.train()

    use_bf16 = torch.cuda.is_bf16_supported()
    ssft_out = WORK_DIR / "ssft_adapter"
    if ssft_out.is_dir():
        shutil.rmtree(ssft_out)
    ssft_out.mkdir(parents=True, exist_ok=True)

    train_args = TrainingArguments(
        output_dir=str(WORK_DIR / "ssft_trainer_state"),
        num_train_epochs=SSFT_EPOCHS,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=SSFT_GRAD_ACCUM,
        learning_rate=SSFT_LR,
        warmup_ratio=SSFT_WARMUP_RATIO,
        lr_scheduler_type="cosine",
        weight_decay=0.01,
        bf16=use_bf16,
        fp16=not use_bf16,
        logging_steps=20,
        save_strategy="no",
        report_to="none",
        max_grad_norm=1.0,
        dataloader_pin_memory=False,
    )

    def collate(features):
        return {
            "input_ids": torch.tensor([f["input_ids"] for f in features], dtype=torch.long),
            "attention_mask": torch.tensor([f["attention_mask"] for f in features], dtype=torch.long),
            "labels": torch.tensor([f["labels"] for f in features], dtype=torch.long),
        }

    trainer = Trainer(
        model=model, args=train_args, train_dataset=tok_ds, data_collator=collate
    )
    trainer.train()
    model.save_pretrained(str(ssft_out))

    shutil.make_archive(str(WORK_DIR / "submission"), "zip", ssft_out)

    meta = {
        "source_adapter_dir": str(adapter_dir),
        "ssft_adapter_dir": str(ssft_out),
        "traces_used": int(len(traces)),
        "per_family": per_fam,
        "epochs": SSFT_EPOCHS,
        "lr": SSFT_LR,
    }
    with open(WORK_DIR / "ssft_meta.json", "w") as f:
        json.dump(meta, f, indent=2)
        f.write("\n")
    print(f"[SSFT] adapter saved -> {ssft_out}")
    print(f"[SSFT] submission.zip updated "
          f"({(WORK_DIR / 'submission.zip').stat().st_size / 1024 / 1024:.2f} MB)")

    del trainer, model, tok_ds, raw_ds, tokenizer
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    shutil.rmtree(ssft_out, ignore_errors=True)
    shutil.rmtree(WORK_DIR / "ssft_trainer_state", ignore_errors=True)


if ENABLE_SOLVER_SFT:
    run_solver_sft(_adapter_dir, WORK_DIR / "submission.zip")
else:
    print("ENABLE_SOLVER_SFT=False -> submission.zip kept as the Direction-1 adapter.")
